# 02 Cleaning

Clean the raw data, standardize column names, add analysis-friendly derived fields, and export a processed dataset to `data/processed/cleaned_dataset.csv`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
RAW_PATH = PROJECT_ROOT / 'data/raw/bank-full.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result.columns = (
        result.columns.str.strip()
        .str.lower()
        .str.replace(r'[^a-z0-9]+', '_', regex=True)
        .str.strip('_')
    )
    return result


def clean_bank_marketing(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = normalize_columns(df)
    cleaned = cleaned.drop_duplicates().reset_index(drop=True)

    for column in cleaned.select_dtypes(include='object').columns:
        cleaned[column] = cleaned[column].astype('string').str.strip().str.lower()

    cleaned['y'] = cleaned['y'].map({'no': 0, 'yes': 1}).astype('int8')
    cleaned['previously_contacted'] = (cleaned['pdays'] != -1).astype('int8')
    cleaned['duration_minutes'] = (cleaned['duration'] / 60).round(2)
    cleaned['age_group'] = pd.cut(
        cleaned['age'],
        bins=[0, 24, 34, 44, 54, 64, 200],
        labels=['18-24', '25-34', '35-44', '45-54', '55-64', '65+'],
        include_lowest=True,
    )
    cleaned['balance_band'] = pd.cut(
        cleaned['balance'],
        bins=[-100000, 0, 1000, 5000, 1000000],
        labels=['negative_or_zero', 'low', 'medium', 'high'],
    )
    cleaned['campaign_band'] = pd.cut(
        cleaned['campaign'],
        bins=[0, 1, 3, 5, 100],
        labels=['1', '2-3', '4-5', '6+'],
        include_lowest=True,
    )

    return cleaned


In [19]:
raw_df = pd.read_csv(RAW_PATH, sep=';')
clean_df = clean_bank_marketing(raw_df)

print(f'Input rows: {len(raw_df):,}')
print(f'Output rows: {len(clean_df):,}')
print(f'Rows removed as duplicates: {len(raw_df) - len(clean_df):,}')
print(f'Total missing values after cleaning: {int(clean_df.isna().sum().sum()):,}')

display(clean_df.head())
display(clean_df.dtypes.to_frame('dtype'))


Input rows: 45,211
Output rows: 45,211
Rows removed as duplicates: 0
Total missing values after cleaning: 0


/var/folders/10/wlq9tyb95b517ntkh99p3wsr0000gp/T/ipykernel_18799/2932537686.py:43: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in cleaned.select_dtypes(include='object').columns:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,previously_contacted,duration_minutes,age_group,balance_band,campaign_band
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,0,0,4.35,55-64,medium,1
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,0,0,2.52,35-44,low,1
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,0,0,1.27,25-34,low,1
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,0,0,1.53,45-54,medium,1
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,0,0,3.30,25-34,low,1


,dtype
age,int64
job,string
marital,string
education,string
default,string
balance,int64
housing,string
loan,string
contact,string
day,int64


In [21]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(PROCESSED_PATH, index=False)

print(f'Saved cleaned dataset to {PROCESSED_PATH}')
display(clean_df['y'].value_counts().rename_axis('y').to_frame('count'))


Saved cleaned dataset to /Users/Harsh/Documents/Github/Sec-B_G-2_BankMarketing/data/processed/cleaned_dataset.csv


,count
y,
0,39922
1,5289
